In [ ]:
# VERICODING ARC-AGI-3 | v17 — TTT + beam + dataset v14 compatible
# Auto-detects dataset path, installs arc-agi offline, runs run_submission()
import subprocess, sys, os, torch
from pathlib import Path

os.environ.setdefault('ARC_API_KEY', 'anon')
os.environ.setdefault('ARC_BASE_URL', 'https://three.arcprize.org')
os.environ.setdefault('OPERATION_MODE', 'offline')
os.environ.setdefault('ENVIRONMENTS_DIR',
    '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files')
os.environ.setdefault('RECORDINGS_DIR', '/kaggle/working/recordings')

DS_CANDIDATES = [
    Path('/kaggle/input/datasets/krisskey/vericoding-urm'),
    Path('/kaggle/input/vericoding-urm'),
    Path('/kaggle/input/datasets/vericoding-urm'),
]
DS = next((p for p in DS_CANDIDATES if p.is_dir() and (p / 'wasm_bridge.py').exists()), None)
if not DS:
    raise SystemExit('[v17] Dataset not found — attach vericoding-urm dataset')
print(f'[v17] Dataset: {DS}')

sys.path.insert(0, str(DS))
for p in [DS / 'external', DS / 'external/urm']:
    if p.is_dir(): sys.path.insert(0, str(p))

COMP = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3')
WHEELS = COMP / 'arc_agi_3_wheels'

# GPU info
if torch.cuda.is_available():
    n = torch.cuda.get_device_name(0)
    v = torch.cuda.get_device_properties(0).total_memory / 1e9
    g = torch.cuda.device_count()
    print(f'[v17] GPU: {n} ({v:.1f}GB) x{g}')
else:
    print('[v17] CPU only')

# Install arc-agi
try:
    import arc_agi; print(f'[v17] arc-agi already installed')
except ImportError:
    cmd = [sys.executable, '-m', 'pip', 'install', 'arc-agi', '-q']
    if WHEELS.is_dir(): cmd += ['--no-index', f'--find-links={WHEELS}']
    subprocess.check_call(cmd)
    import arc_agi; print('[v17] arc-agi installed ok')

# Run
from kaggle_main import run_submission
run_submission()
